# 三个可交互的直觉实验

这是《AI by Intuition》的 **P2 补充练习**。每个实验单独调整一个变量，观察数学结果；不需要 GPU、私有数据或下载模型权重。

1. **Softmax 温度**：只改变分数被放大/压缩的程度，排序不变（有限正温度），概率分布会变。
2. **Attention 读取比例**：使用教材三维主例的因果注意力分数近似值，观察遮挡与行内归一化。
3. **梯度下降学习率**：沿用 P0 的三点回归，比较稳定下降、震荡和发散。

图示用于直觉；完整三维 Transformer 的权重与损失数值仍以仓库 `scripts/teaching_reference.py` 为准。

In [ ]:
%pip install -q numpy matplotlib ipywidgets

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def stable_softmax(scores, temperature=1.0):
    """Softmax(z/tau) with the max subtracted for numerical stability."""
    if temperature <= 0:
        raise ValueError("temperature must be positive")
    z = np.asarray(scores, dtype=np.float64) / float(temperature)
    e = np.exp(z - z.max())
    return e / e.sum()

assert np.isclose(stable_softmax([0, np.log(2), 0])[1], 0.5)
print("三分类基线正确：p(第二类) = 0.5")

## 实验 1：Softmax 温度

固定三个 logits 为 `[0, ln(2), 0]`：温度为 1 时，概率为 `[0.25, 0.50, 0.25]`，正确类的交叉熵为 `ln(2)≈0.6931`。温度大于 1 会让概率更平缓；温度接近 0 时会更集中。这里不更改 logits，也没有重新训练模型。

In [ ]:
def show_temperature(temperature=1.0):
    logits = np.array([0.0, np.log(2.0), 0.0])
    p = stable_softmax(logits, temperature)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(["Class 0", "Class 1", "Class 2"], p)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Softmax probability")
    ax.set_title(f"Temperature={temperature:.2f}; target CE={-np.log(p[1]):.4f}")
    plt.show()
    print("概率:", np.round(p, 6), "和=", round(float(p.sum()), 8))

widgets.interact(
    show_temperature,
    temperature=widgets.FloatSlider(value=1.0, min=0.2, max=3.0,
                                    step=0.1, description="Temperature")
);

## 实验 2：因果 Attention 读取权重

使用教材初始三维主例的 **3×3 因果打分矩阵（展示用四位小数）**，其中未来位置始终不可见。拖动 Query 位置和温度；观察某行只有前 `i` 个 Key 有权重，Softmax 的该行总和始终为 1。温度滑块只做局部对照，**不是重新运行完整模型**。

In [ ]:
scores = np.array([
    [1.0583, np.nan, np.nan],
    [-0.9019, 1.0496, np.nan],
    [-0.4713, -0.1071, 0.7855],
], dtype=np.float64)

def masked_attention(temperature):
    a = np.zeros((3, 3), dtype=np.float64)
    for i in range(3):
        a[i, :i+1] = stable_softmax(scores[i, :i+1], temperature)
    return a

a_reference = masked_attention(1.0)
assert np.allclose(a_reference.sum(axis=1), 1.0)
assert np.allclose(a_reference[1, :], [0.1244, 0.8756, 0.0], atol=5e-5)

def show_attention(query=3, temperature=1.0):
    a = masked_attention(temperature)
    fig, ax = plt.subplots(figsize=(5.5, 3.6))
    ax.imshow(a, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(3), ["Key 1", "Key 2", "Key 3"])
    ax.set_yticks(range(3), ["Query 1", "Query 2", "Query 3"])
    for i in range(3):
        for j in range(3):
            label = f"{a[i,j]:.3f}" if j <= i else "Mask"
            ax.text(j, i, label, ha="center", va="center",
                    color="white" if a[i,j] > 0.5 else "black")
    ax.add_patch(plt.Rectangle((-0.5, query-1.5), 3, 1,
                               edgecolor="orange", linewidth=2, fill=False))
    ax.set_title(f"Query {query}: visible-key weight sum = {a[query-1].sum():.3f}")
    plt.show()

widgets.interact(
    show_attention,
    query=widgets.IntSlider(value=3, min=1, max=3, description="Query"),
    temperature=widgets.FloatSlider(value=1.0, min=0.2, max=3.0,
                                    step=0.1, description="Temperature")
);

## 实验 3：学习率对梯度下降的影响

沿用同样的三点 `(-1,-1), (0,1), (1,3)`，模型为 `ŷ=wx+b`，初值 `(w,b)=(1,0)`。损失为 `L(w,b)=2/3·(w−2)²+(b−1)²`，两项参数同步用相同旧梯度更新。**学习率 0.3** 时一步后参数为 `(1.4,0.6)`，损失由 `5/3` 降为 `0.4`。大的学习率可能振荡甚至发散。

In [ ]:
def mse_loss(w, b):
    return 2.0 / 3.0 * (w - 2.0)**2 + (b - 1.0)**2

def gradient_descent(learning_rate, steps=20):
    w, b = 1.0, 0.0
    losses = [mse_loss(w, b)]
    for _ in range(steps):
        gw, gb = 4.0 / 3.0 * (w - 2.0), 2.0 * (b - 1.0)
        w, b = w - learning_rate * gw, b - learning_rate * gb
        losses.append(mse_loss(w, b))
    return losses, w, b

reference, _, _ = gradient_descent(0.3, steps=1)
assert np.isclose(reference[0], 5/3) and np.isclose(reference[1], 0.4)

def show_learning_rate(learning_rate=0.3):
    losses, w, b = gradient_descent(learning_rate)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(range(len(losses)), losses, marker=".", label="Training MSE")
    ax.set_xlabel("SGD steps")
    ax.set_ylabel("Three-point MSE")
    ax.set_title(f"η={learning_rate:.2f}, 最后 L={losses[-1]:.4f}")
    ax.grid(True, alpha=0.3)
    plt.show()
    print(f"起点 L={losses[0]:.6f}; 第一步 L={losses[1]:.6f}; "
          f"20 步后 (w,b)=({w:.4f},{b:.4f})")

widgets.interact(
    show_learning_rate,
    learning_rate=widgets.FloatSlider(value=0.3, min=0.05, max=1.2,
                                      step=0.05, description="学习率 η")
);

## 自测

- 温度从 `1` 改为 `2` 时，最大 logit 对应的类别是否改变？正确类的交叉熵是否改变？
- Query 2 为什么无论怎么调温度都无法读取 Key 3？
- 将学习率从 `0.3` 改为 `1.1`，损失曲线为什么可能不再下降？

**检查方法：**保持正确类标签、Mask 和训练数据不变，每次只改变一个滑块；先解释再看图。Notebook 不改动原书的三维 Transformer 数值基线。